# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template and demonstration for loading and exploring the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets and their fields, using the `@id` of each entity.

Let's list all available record sets in the dataset, along with their `@id` and field `@id`s.

In [ ]:
# List record sets and their field @id's
print("Available record sets and fields (by @id):\n")

record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # Fallback for possible alternate attribute
    record_sets = getattr(metadata, 'recordSet', [])

record_set_ids = []
for record_set in record_sets:
    print(f"RecordSet name: {getattr(record_set, 'name', None)} | @id: {getattr(record_set, '@id', None)}")
    fields = getattr(record_set, 'fields', [])
    print('  Field @ids:')
    for field in fields:
        print(f"    - {getattr(field, 'name', None)} | @id: {getattr(field, '@id', None)}")
    record_set_ids.append(record_set['@id'] if isinstance(record_set, dict) and '@id' in record_set else getattr(record_set, '@id', None))

if not record_set_ids:
    # Try to extract RecordSet from underlying JSON if possible
    from collections import abc
    if hasattr(metadata, 'to_json'):
        meta_json = metadata.to_json()
        recsets = meta_json.get('recordSet') or meta_json.get('record_sets') or []
        for recset in recsets:
            recid = recset.get('@id')
            record_set_ids.append(recid)
            print(f"RecordSet @id: {recid}")
            if 'field' in recset:
                print('  Field @ids:')
                fields = recset['field']
                if not isinstance(fields, list):
                    fields = [fields]
                for field in fields:
                    # field is either a dict or @id string
                    if isinstance(field, str):
                        print(f"    - {field}")
                    elif isinstance(field, dict):
                        print(f"    - {field.get('@id', '')}")
if not record_set_ids:
    print('No record sets found in metadata.')

## 3. Data Extraction
Load all available record sets into pandas DataFrames using their `@id`.

In [ ]:
# If you know the record set @id(s) from section 2, add them here.
# For this dataset, extract them similarly as in previous cell.

# Try to extract record sets from metadata (repeat from above if not cached)
rs_ids = record_set_ids if 'record_set_ids' in locals() and record_set_ids else []
if not rs_ids:
    meta_json = metadata.to_json()
    recsets = meta_json.get('recordSet', [])
    rs_ids = [rec.get('@id') for rec in recsets] if recsets else []

# Load records from each record set into a DataFrame
dataframes = {}
for rsid in rs_ids:
    print(f"Loading records for RecordSet @id: {rsid}")
    try:
        recs = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(recs)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} records. Columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"Failed to load records for {rsid}: {e}")

if dataframes:
    # Show columns and preview for the first available record set
    first_rs = list(dataframes.keys())[0]
    print(f"Available columns in RecordSet {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data exploration steps: filtering, normalization, grouping, etc., using field `@id` as column names.

In [ ]:
import numpy as np
# For demonstration, select the first available record set and inspect for numeric columns
if dataframes:
    first_rs = list(dataframes.keys())[0]
    df = dataframes[first_rs]
    # Pick the first numeric field as example
    numeric_field = None
    for col in df.columns:
        # Heuristics for numeric columns
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
        # Try to cast to float
        try:
            df[col] = pd.to_numeric(df[col])
            if np.issubdtype(df[col].dtype, np.number):
                numeric_field = col
                break
        except:
            continue
    if numeric_field is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field '{numeric_field}' (@id) for exploration.")
        # Filter: keep only values above a threshold (e.g. 10)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold}:")
        print(filtered_df.head())
        # Normalize that column
        filtered_df[numeric_field + '_normalized'] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())
        # Try grouping by another field, e.g. by first non-numeric string column
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean').reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or field relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution and a boxplot for the numeric field
if dataframes and 'numeric_field' in locals() and numeric_field is not None:
    plt.figure(figsize=(10,4))
    # Distribution
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    # Boxplot
    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")
    plt.tight_layout()
    plt.show()

    # If grouping, plot group means
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped_df[group_field], y=grouped_df['mean'])
        plt.title(f"Mean {numeric_field} per {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook we loaded the Croissant dataset, explored available record sets and fields by `@id`, extracted the data, conducted basic filtering and normalization, grouped records for summary statistics, and visualized the main distributions. 

To continue exploring, try using mlcroissant to work with additional record sets, experiment with more advanced filters, or build models on the data!